# 02 Risk Analytics and Stress Testing

## Project context

This notebook builds Phase 1 of the Fixed Income and Balance Sheet Analytics Platform. Phase 0 created a synthetic bond book, cash flow schedule, pricing outputs, and portfolio market value summary. Phase 1 adds duration, convexity, DV01, parallel rate shocks, and a simple balance sheet interpretation.

The bond book is synthetic and for portfolio learning only.

## Risk analytics objective

- Calculate bond-level Macaulay duration, modified duration, convexity, and DV01.
- Summarize portfolio-level interest-rate sensitivity.
- Reprice the synthetic bond book under parallel rate shocks.
- Interpret stress losses by sector and rating.
- Create simple ALM-style notes for balance sheet sensitivity.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR, OUTPUTS_DIR, FIGURES_DIR
from src.risk import calculate_bond_risk_metrics, summarize_portfolio_risk
from src.scenarios import (
    create_rate_scenarios,
    run_stress_test,
    summarize_stress_results,
    create_simple_alm_summary,
)
from src.visualization import (
    plot_duration_by_bond,
    plot_convexity_by_bond,
    plot_dv01_by_bond,
    plot_portfolio_value_under_shocks,
    plot_stress_loss_by_group,
)

scenario_dir = OUTPUTS_DIR / "scenarios"
scenario_dir.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

settlement_date = pd.Timestamp("2026-04-30")

## Load bond book and pricing results

In [ ]:
bond_book = pd.read_csv(PROCESSED_DATA_DIR / "synthetic_bond_book.csv", parse_dates=["issue_date", "maturity_date"])
cashflows = pd.read_csv(OUTPUTS_DIR / "bond_book" / "bond_cashflows.csv", parse_dates=["settlement_date", "cashflow_date"])
pricing_results = pd.read_csv(OUTPUTS_DIR / "pricing" / "bond_pricing_results.csv", parse_dates=["issue_date", "maturity_date", "settlement_date"])
portfolio_pricing_summary = pd.read_csv(OUTPUTS_DIR / "pricing" / "portfolio_pricing_summary.csv")

bond_book.shape, cashflows.shape, pricing_results.shape, portfolio_pricing_summary

## Duration and convexity calculation

In [ ]:
risk_results = calculate_bond_risk_metrics(pricing_results, cashflows)
risk_results.to_csv(scenario_dir / "bond_risk_metrics.csv", index=False)
risk_results

## DV01 calculation

DV01 estimates the market value change for a one basis point rate move. The sign convention here reports positive exposure amount; a rate increase would reduce bond value approximately by DV01 per basis point before convexity adjustment.

In [ ]:
risk_results[["bond_id", "market_value", "modified_duration", "convexity", "dv01"]].sort_values("dv01", ascending=False)

## Portfolio risk summary

In [ ]:
portfolio_risk_summary = summarize_portfolio_risk(risk_results)
portfolio_risk_summary.to_csv(scenario_dir / "portfolio_risk_summary.csv", index=False)
portfolio_risk_summary.T

## Interest-rate stress scenarios

In [ ]:
rate_scenarios = create_rate_scenarios()
rate_scenarios

In [ ]:
stress_results = run_stress_test(
    bond_book=bond_book,
    shocks_bps=rate_scenarios["shock_bps"].tolist(),
    settlement_date=settlement_date,
)
stress_results.to_csv(scenario_dir / "rate_stress_results.csv", index=False)
stress_results.head()

## Price impact by bond and portfolio

In [ ]:
stress_summary = summarize_stress_results(stress_results)
stress_summary.to_csv(scenario_dir / "rate_stress_summary.csv", index=False)
stress_summary

## Sector and rating risk concentration

In [ ]:
sector_stress = (
    stress_results[stress_results["shock_bps"] == 100]
    .groupby("sector", as_index=False)["market_value_change"]
    .sum()
    .assign(stress_loss=lambda df: -df["market_value_change"])
    .sort_values("stress_loss", ascending=False)
)
rating_stress = (
    stress_results[stress_results["shock_bps"] == 100]
    .groupby("credit_rating", as_index=False)["market_value_change"]
    .sum()
    .assign(stress_loss=lambda df: -df["market_value_change"])
    .sort_values("stress_loss", ascending=False)
)
sector_stress, rating_stress

## Simple ALM / balance sheet interpretation

In [ ]:
alm_summary = create_simple_alm_summary(
    portfolio_market_value=float(portfolio_risk_summary["total_market_value"].iloc[0]),
    stress_summary=stress_summary,
)
alm_summary.to_csv(scenario_dir / "simple_alm_summary.csv", index=False)
alm_summary

## Figures

In [ ]:
fig = plot_duration_by_bond(risk_results)
fig.savefig(FIGURES_DIR / "duration_by_bond.png", dpi=150, bbox_inches="tight")
fig

In [ ]:
fig = plot_convexity_by_bond(risk_results)
fig.savefig(FIGURES_DIR / "convexity_by_bond.png", dpi=150, bbox_inches="tight")
fig

In [ ]:
fig = plot_dv01_by_bond(risk_results)
fig.savefig(FIGURES_DIR / "dv01_by_bond.png", dpi=150, bbox_inches="tight")
fig

In [ ]:
fig = plot_portfolio_value_under_shocks(stress_summary)
fig.savefig(FIGURES_DIR / "portfolio_value_under_rate_shocks.png", dpi=150, bbox_inches="tight")
fig

In [ ]:
fig = plot_stress_loss_by_group(stress_results, group_col="sector", shock_bps=100)
fig.savefig(FIGURES_DIR / "stress_loss_by_sector.png", dpi=150, bbox_inches="tight")
fig

## Business interpretation

The synthetic portfolio has positive duration exposure, so parallel upward rate shocks reduce market value while downward shocks increase it. DV01 gives a compact balance sheet sensitivity measure: it estimates how much value changes for a one basis point move. Sector and rating cuts help identify where rate sensitivity is concentrated before adding real ALM assumptions.

## Limitations

- Synthetic bond book only.
- Parallel rate shocks only.
- No real yield curve calibration.
- No liquidity, credit spread, prepayment, or optionality modeling.
- ALM interpretation is simplified and does not include liabilities.

## Final project outputs

Phase 1 adds `bond_risk_metrics.csv`, `portfolio_risk_summary.csv`, `rate_stress_results.csv`, `rate_stress_summary.csv`, and `simple_alm_summary.csv` under `outputs/scenarios/`.

## Next steps for Phase 2 polish

- Polish the README and reports.
- Add resume bullets and interview talking points.
- Make limitations clear for recruiters and finance interviewers.